### Project By- Pranali for DASA TECHNOWORLD PVT. LTD.

# Feature Engineering

# Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/farm_ml_clean_data.csv")
df.head()

,Farmer_ID,State,Farm_Size_Birds,Experience_Years,Join_Date,Contract_Type,Contract_Duration_Months,Expected_ROI_%,Total_Chicks,Total_Mortality,Total_Feed_Consumed,Total_Feed_Used_KG,Total_Feed_Cost,Total_Sales_Qty,Total_Revenue,Mortality_Rate,Profit
0,100000,UP,9042,8,2022-01-31,Revenue_Sharing,12.0,16.0,4104.0,33.0,4101.0,8100.0,255839.0,8191.0,817700.0,0.008041,396953.5
1,100001,Maharashtra,17015,2,2023-06-07,Revenue_Sharing,12.0,16.0,1178.0,101.0,12020.0,12871.0,368959.0,5592.0,675278.0,0.085739,396953.5
2,100002,Tamil Nadu,14906,14,2023-07-06,Buy_Back,12.0,12.0,3770.0,400.0,24500.0,8100.0,255839.0,10752.0,1012352.0,0.106101,396953.5
3,100003,Telangana,706,19,2025-10-27,Revenue_Sharing,12.0,16.0,1754.0,281.0,4722.0,25057.0,850025.0,5978.0,974851.0,0.160205,124826.0
4,100004,Maharashtra,8072,16,2024-06-15,Buy_Back,18.0,10.0,12844.0,318.0,21365.0,3166.0,51899.0,16567.0,1760982.0,0.024759,1709083.0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22917 entries, 0 to 22916
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Farmer_ID                 22917 non-null  int64  
 1   State                     22917 non-null  str    
 2   Farm_Size_Birds           22917 non-null  int64  
 3   Experience_Years          22917 non-null  int64  
 4   Join_Date                 22917 non-null  str    
 5   Contract_Type             22917 non-null  str    
 6   Contract_Duration_Months  22917 non-null  float64
 7   Expected_ROI_%            22917 non-null  float64
 8   Total_Chicks              22917 non-null  float64
 9   Total_Mortality           22917 non-null  float64
 10  Total_Feed_Consumed       22917 non-null  float64
 11  Total_Feed_Used_KG        22917 non-null  float64
 12  Total_Feed_Cost           22917 non-null  float64
 13  Total_Sales_Qty           22917 non-null  float64
 14  Total_Revenue    

# Target Creation

In [3]:
# Create Farm Performance Target

# Business Thresholds
GOOD_PROFIT = 50000
AVERAGE_PROFIT = 20000
LOW_MORTALITY = 0.05

def classify_farm(row):
    """Classify farm performance based on Profit and Mortality Rate."""

    if (row["Profit"] >= GOOD_PROFIT) and (row["Mortality_Rate"] <= LOW_MORTALITY):
        return "GOOD_PERFORMANCE"

    elif row["Profit"] >= AVERAGE_PROFIT:
        return "AVERAGE_PERFORMANCE"

    else:
        return "POOR_PERFORMANCE"


# Create Target Column
df["Performance"] = df.apply(classify_farm, axis=1)

# Check Target Distribution
print("Farm Performance Distribution:\n")
print(df["Performance"].value_counts())

# Preview
df[["Profit", "Mortality_Rate", "Performance"]].head()

Farm Performance Distribution:

Performance
AVERAGE_PERFORMANCE    12683
GOOD_PERFORMANCE        6634
POOR_PERFORMANCE        3600
Name: count, dtype: int64


,Profit,Mortality_Rate,Performance
0,396953.5,0.008041,GOOD_PERFORMANCE
1,396953.5,0.085739,AVERAGE_PERFORMANCE
2,396953.5,0.106101,AVERAGE_PERFORMANCE
3,124826.0,0.160205,AVERAGE_PERFORMANCE
4,1709083.0,0.024759,GOOD_PERFORMANCE


# Feature Selection

In [4]:
# Feature Selection for Farm Performance Model

# Keep a copy of the complete feature engineered dataset
df_master = df.copy()

df_master.to_csv(
    "../data/processed/farm_feature_engineered.csv",
    index=False
)
# Columns removed only for Farm Performance prediction
drop_cols = [
    "Farmer_ID",
    "Join_Date",
    "Profit",
    "Total_Revenue",
    "Total_Feed_Cost"
]

df_model = df.drop(columns=drop_cols)

print("Farm Performance Dataset Shape:", df_model.shape)

display(df_model.head())

Farm Performance Dataset Shape: (22917, 13)


,State,Farm_Size_Birds,Experience_Years,Contract_Type,Contract_Duration_Months,Expected_ROI_%,Total_Chicks,Total_Mortality,Total_Feed_Consumed,Total_Feed_Used_KG,Total_Sales_Qty,Mortality_Rate,Performance
0,UP,9042,8,Revenue_Sharing,12.0,16.0,4104.0,33.0,4101.0,8100.0,8191.0,0.008041,GOOD_PERFORMANCE
1,Maharashtra,17015,2,Revenue_Sharing,12.0,16.0,1178.0,101.0,12020.0,12871.0,5592.0,0.085739,AVERAGE_PERFORMANCE
2,Tamil Nadu,14906,14,Buy_Back,12.0,12.0,3770.0,400.0,24500.0,8100.0,10752.0,0.106101,AVERAGE_PERFORMANCE
3,Telangana,706,19,Revenue_Sharing,12.0,16.0,1754.0,281.0,4722.0,25057.0,5978.0,0.160205,AVERAGE_PERFORMANCE
4,Maharashtra,8072,16,Buy_Back,18.0,10.0,12844.0,318.0,21365.0,3166.0,16567.0,0.024759,GOOD_PERFORMANCE


# One-Hot Encoding

In [5]:
# Encode Categorical Variables
categorical_cols = ['State', 'Contract_Type']

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Label Encoding

In [6]:
from sklearn.preprocessing import LabelEncoder
import joblib

# Encode Target Variable
label_encoder = LabelEncoder()

df["Farm_Performance"] = label_encoder.fit_transform(df["Performance"])
df_model["Farm_Performance"] = df["Performance"]

# Label Mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print("Label Encoding Mapping:\n")
print(label_mapping)

Label Encoding Mapping:

{'AVERAGE_PERFORMANCE': np.int64(0), 'GOOD_PERFORMANCE': np.int64(1), 'POOR_PERFORMANCE': np.int64(2)}


# Feature Scaling

In [7]:
# Scale Numerical Features
num_cols = [
    'Farm_Size_Birds',
    'Experience_Years',
    'Total_Chicks',
    'Total_Mortality',
    'Total_Feed_Consumed',
    'Total_Sales_Qty',
    'Expected_ROI_%'
]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_model[num_cols] = scaler.fit_transform(df_model[num_cols])

# Create Features and Target

In [8]:
X = df_model.drop(columns=["Performance"])
y = df_model["Performance"]

print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Matrix Shape : (22917, 20)
Target Shape : (22917,)


In [9]:
print("\nFinal Feature Names:")

for col in X.columns:
    print(col)


Final Feature Names:
Farm_Size_Birds
Experience_Years
Contract_Duration_Months
Expected_ROI_%
Total_Chicks
Total_Mortality
Total_Feed_Consumed
Total_Feed_Used_KG
Total_Sales_Qty
Mortality_Rate
State_Karnataka
State_Maharashtra
State_Ontario
State_Queensland
State_Tamil Nadu
State_Telangana
State_Texas
State_UP
Contract_Type_Revenue_Sharing
Farm_Performance


# Save X and y

In [10]:
# Farm Performance Dataset
X.to_csv(
    "../data/processed/X_features.csv",
    index=False
)

y.to_frame(name="Farm_Performance").to_csv(
    "../data/processed/y_target.csv",
    index=False
)

print("All datasets saved successfully.")

All datasets saved successfully.


# Save Preprocessors

In [11]:
import joblib

feature_columns = X.columns.tolist()

joblib.dump(
    label_encoder,
    "../models/label_encoder.pkl"
)

joblib.dump(
    feature_columns,
    "../models/farm_feature_columns.pkl"
)

print("Preprocessors saved successfully.")

Preprocessors saved successfully.
